**temporary notebook (editor)**

build response matrix Y for each week

then, for each week Y matrix, put it into a dataframe with cols: subj_id, week, v{i} (i in [0, p-1] where p = x * y * z)

In [ ]:
import numpy as np
import pandas as pd
import nibabel as nib
import nitools as nt
import os

import smarts_cerebellum.globals as gl
from smarts_cerebellum.util import subj_path_search

In [31]:
def world_indices(img):
    """
    Function to get indices (x,y,z) for world coordinates for an image
    """
    i, j, k = np.indices(img.shape)
    x,y,z = nt.affine_transform(i, j, k, img.affine)
    return x,y,z

In [ ]:
def response_matrix_week(subj_path_dict,
                         x,y,z, # indices from world image
                         week, # which week this matrix is for
                         ):
    # initialize empty array
    week_subj_rows = []
    week_subj_ids = [] # extra check: store subj_id AFTER they are added to matrix
    
    for s_id, s_path in subj_path_dict.items():
        # let's do a try-except loop
        try:
            subj_img = nib.load(s_path)
            row = nt.sample_image(subj_img, 
                                  xm = x, ym = y, zm = z, # resample to template world indices - pass to fcn template world indices
                                  interpolation = 1
                                  ).flatten() # store as row vector
        except Exception: # file path not exist; subj not added to s_id for that week
            print(f'path {s_path} not exist; skip')
            continue

        week_subj_rows.append(row)
        week_subj_ids.append(s_id)
    
    Y_w = np.array(week_subj_rows)
    week_subj_ids = np.array(week_subj_ids)

    return Y_w,week_subj_ids, week # returns week used

In [ ]:
def make_week_dataframe(
                   subj_path_dict, 
                   week, # which week this df is for
                   template_img):
    """
    makes dataframe out of each week's response matrix
    """
    x,y,z, = world_indices(template_img)

    Y_w, week_subjs, week = response_matrix_week(subj_path_dict,
                                           x,y,z,
                                           week # returns same week
                                           )
    
    template_arr = template_img.get_fdata()
    P = np.prod(template_arr.shape)


    df = pd.DataFrame(data = Y_w, 
                      columns = [f'v{i}' for i in range(P)]
                      )
    df.insert(0, 'Week', week) # add week as beginning col
    df.insert(0, 'Subj', week_subjs)

    return df

function to make a response dataframe

function to perform the lme (combines everything) - make sure this uses world_indices

In [ ]:
def response_dataframe(subj_path_dict_list, # each week's dictionary
            
                       template_img,
                       #time_pts = [0, 4, 12, 24, 52], # all time points
                       ):
    """
    Make full response dataframe (concatenate week dataframes)
    """
    dfs = []
    for idx, t in enumerate(time_pts):
        week_df = make_week_dataframe(subj_path_dict = subj_path_dict_list[idx], # get i^th week's dictionary
                                      week = t,
                                      template_img = template_img,
                                      )
        dfs.append(week_df)

    Y_df = pd.concat(dfs, axis = 0, ignore_index = True)
    return Y_df

In [2]:
p_df = pd.read_csv(os.path.join(gl.baseDir, 'participants_anat.tsv'), sep = '\t')

In [3]:
small_df = p_df.iloc[:11]

In [4]:
small_df

,SN,ID,Centre,Week,week,RefT1,numrun,nslices,Hand,LesionSide,...,include1,lesiondef,behavior_missing,behavior_blocks,has_mvc,DTImap_missing,CentreNo,machine,subj_id,keep
0,1,2310,CU,W0,0,W0,8,35,b,left,...,1,1,0,8,0,0,1,naveed,CU_2310,True
1,2,2310,CU,W4,4,W0,8,35,b,left,...,1,1,0,8,0,0,1,naveed,CU_2310,True
2,3,2310,CU,W12,12,W0,8,35,b,left,...,1,1,0,8,1,0,1,naveed,CU_2310,True
3,4,2310,CU,W24,24,W0,8,35,b,left,...,1,1,0,8,1,0,1,naveed,CU_2310,True
4,5,2310,CU,W52,52,W0,8,35,b,left,...,1,1,3,0,1,0,1,naveed,CU_2310,True
5,6,2538,CU,W0,0,W0,8,35,b,right,...,1,1,0,8,1,0,1,naveed,CU_2538,True
6,7,2538,CU,W4,4,W0,8,35,b,right,...,1,1,0,8,1,0,1,naveed,CU_2538,True
7,8,2663,CU,W0,0,W0,7,35,b,left,...,1,1,0,7,1,0,1,naveed,CU_2663,True
8,9,2663,CU,W4,4,W0,8,35,b,left,...,1,1,0,8,1,0,1,naveed,CU_2663,True
9,10,2663,CU,W12,12,W0,8,35,b,left,...,1,1,3,0,1,0,1,naveed,CU_2663,True


In [ ]:
def make_week_dicts(df,
                    ref_subj = 'CU_2310',
                    #time_points = [0, 4, 12, 24, 52],
                    subdir = 'MNISym_T1',
                    file_suffix = 'MNISym_T1_coreg_reslice.nii.gz',
                    ):
    dictionaries = []
    for week in time_points:
        ref_search_path = os.path.join(gl.baseDir, subdir, ref_subj, f'{ref_subj}_W{week}_{file_suffix}')
        paths, subjs = subj_path_search(ref_search_path, ref_subj, week, df)
        dictionaries.append(dict(zip(subjs, paths)))
    return dictionaries

**for timepoints, define it in the wrapper function**
take out of: make_week_dicts, resopnse_dataframe

In [38]:
template_img = '/home/UWO/mporwal2/Documents/GitHub/smarts_cerebellum/tpl-MNI152NLin2009cSymC_T1w.nii'
template_img = nib.load(template_img)

In [39]:
x,y,z = world_indices(template_img)

In [45]:
y_trial, subj_arr, curr_week = response_matrix_week(dictionary, x,y, z, week)

In [50]:
y_trial.max()

np.float64(2113.4944285456286)

In [55]:
week

24

In [59]:
trial_df = make_week_dataframe(dictionary, week, template_img)

In [62]:
trial_df

,Subj,Week,v0,v1,v2,v3,v4,v5,v6,v7,...,v1208690,v1208691,v1208692,v1208693,v1208694,v1208695,v1208696,v1208697,v1208698,v1208699
0,CU_2310,24,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,CU_2663,24,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,CU_2925,24,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,JHU_2282,24,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,JHU_2395,24,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [63]:
P = np.prod(template_img.shape)

In [64]:
P

np.int64(1208700)

In [ ]:
# need dictionary to include all weeks

In [99]:
y_df_trial = response_dataframe([dictionary1, dictionary2],
           
                       template_img,
                       time_pts = [0, 24], # all time points
                       )

In [ ]:
voxel_cols = [c for c in df.columns if c.startswith('v')]
max_val = df[voxel_cols].values.max()
print(max_val)

In [103]:
v_cols = [v for v in y_df_trial.columns if v.startswith('v')]
max_val = y_df_trial[v_cols].values.max()
print(max_val)

2268.0503201891565
